In [ ]:
import sys
import torch
import pickle
import os
from tqdm.notebook import tqdm

sys.path.insert(0, '..')
sys.path.insert(0, '../../')
sys.path.insert(0, '../../../')
sys.path.insert(0, '../../../../')
sys.path.insert(0, '../../../../../../')

from joinLSTM.model import FullShared_Join_LSTM
from robustness.camargo_evaluation import evaluate_seq_processing
from robustness.robustness_metrics import save_chunk



ModuleNotFoundError: No module named 'robustness.robustness_metrics'

In [8]:
# Load model
file_path_model = '../notebooks/training/Helpdesk/Helpdesk_camargo_leon.pkl'
output_dir = '../../../../evaluation_results/camargo/helpdesk/redo_activity/'
model = FullShared_Join_LSTM.load(file_path_model)

# Load datasets
file_path_original = '../../../../encoded_data/helpdesk/helpdesk_all_5_train.pkl'
#file_path_perturbed = '../../../../encoded_data/helpdesk_large_perturbations_test.pkl'
file_path_perturbed = '../../../../encoded_data/helpdesk/helpdesk_all_5_train.pkl'
#file_path_perturbed = '../../../../encoded_data/helpdesk_perturbed_5_test.pkl'
#file_path_perturbed = '../../../../encoded_data/helpdesk_all_5_test.pkl'
file_path_redo_activity = '../../../../encoded_data/helpdesk/val.pkl'
file_path_redo_activity_pert = '../../../../encoded_data/helpdesk/redo_activity.pkl'


original_dataset = torch.load(file_path_original, weights_only=False)
perturbed_dataset = torch.load(file_path_perturbed, weights_only=False)

redo_activity_dataset = torch.load(file_path_redo_activity, weights_only=False)
redo_activity_pert_dataset = torch.load(file_path_redo_activity_pert, weights_only=False)


print(f"Original dataset loaded: {len(original_dataset)} cases")
print(f"Perturbed dataset loaded: {len(perturbed_dataset)} cases")

FileNotFoundError: [Errno 2] No such file or directory: '../notebooks/training/Helpdesk/Helpdesk_camargo_leon.pkl'

In [ ]:
# Import robustness metrics module
import importlib
import robustness.robustness_metrics
importlib.reload(robustness.robustness_metrics)
from robustness.robustness_metrics import save_chunk

print("Robustness metrics module imported")


Robustness metrics module imported


In [2]:
#create models


from ml_models.reimplemented_comparable_approaches.camargo_LSTM_suffix_pred.robustness.camargo_evaluation import evaluate_with_predefined_prefixes


os.makedirs(output_dir, exist_ok=True)

save_every = 50
results = {}

# Create evaluation generators
# eval_original = evaluate_seq_processing(
#     model=model,
#     dataset=original_dataset,
#     device=torch.device("cpu"),
#     samples_per_case=20,
#     random_order=False
# )

# eval_perturbed = evaluate_seq_processing(
#     model=model,
#     dataset=original_dataset,
#     device=torch.device("cpu"),
#     samples_per_case=20,
#     random_order=False
# )
evaluate_with_predefined_prefixes_normal = evaluate_with_predefined_prefixes(
    model=model,
    dataset=original_dataset,  # Still needed for encoder_decoder and categories
    predefined_pairs=redo_activity_dataset,
    device=torch.device("cpu"),
    samples_per_case=20,
    random_order=False
)

evaluate_with_predefined_prefixes_pert = evaluate_with_predefined_prefixes(
    model=model,
    dataset=original_dataset,  # Still needed for encoder_decoder and categories
    predefined_pairs=redo_activity_pert_dataset,
    device=torch.device("cpu"),
    samples_per_case=20,
    random_order=False
)


print("Evaluation generators created")

ModuleNotFoundError: No module named 'ml_models'

In [3]:
# Main evaluation loop

for i, ((case_name_orig, prefix_len_orig, prefix_orig, sampled_cets_orig, suffix_orig, mean_cet_orig),
        (case_name_pert, prefix_len_pert, prefix_pert, sampled_cets_pert, suffix_pert, mean_cet_pert)) in enumerate(
        tqdm(zip(evaluate_with_predefined_prefixes_normal, evaluate_with_predefined_prefixes_pert), 
        desc="Evaluating robustness")):

    
    #Store results
    key = (case_name_orig, prefix_len_orig)
    results[key] = {
        'original': (prefix_orig, suffix_orig, mean_cet_orig, sampled_cets_orig),
        'perturbed': (prefix_pert, suffix_pert, mean_cet_pert, sampled_cets_pert)
    }
    
    if (i + 1) % save_every == 0:
        save_chunk(results, i, output_dir)
        results = {}

if len(results):
    save_chunk(results, i, output_dir)

print("Robustness evaluation completed!")

NameError: name 'evaluate_with_predefined_prefixes_normal' is not defined

In [4]:
# Load all saved chunks and combine them
all_results = {}
# Get all chunk files and sort them
chunk_files = [f for f in os.listdir(output_dir) if f.startswith('robustness_results_part_')]
chunk_files.sort()  # Ensure correct order

print(f"Found {len(chunk_files)} chunk files")

for chunk_file in chunk_files:
    chunk_path = os.path.join(output_dir, chunk_file)
    print(f"Loading {chunk_file}...")
    with open(chunk_path, 'rb') as f:
        chunk_results = pickle.load(f)
        all_results.update(chunk_results)
        print(f"  Added {len(chunk_results)} results from {chunk_file}")

# Also add the final results if any (e.g. from a still-running evaluation loop)
if 'results' in locals() and len(results) > 0:
    print(f"Adding final {len(results)} results...")
    all_results.update(results)

print(f"\nTotal results loaded: {len(all_results)}")

# Save combined results into a single pickle file
combined_results_path = os.path.join(output_dir, 'robustness_results.pkl')
with open(combined_results_path, 'wb') as f:
    pickle.dump(all_results, f)

print(f"Combined results saved to {combined_results_path}")



NameError: name 'output_dir' is not defined